# Fetch and Display One DIA Cutout

Fetch a single Science / Template / Difference cutout from the Fink broker
for a chosen `diaObjectId`, and overlay:
- cardinal direction arrows (N, E) and the DCR-expected Zenith direction,
- a **dipole axis segment** derived from `r:dipoleAngle`, which is the angle
  of the dipole axis measured from North toward East (standard astronomical
  position angle convention).

**Colourmap convention:**
- Science / Template images: grey scale with a shared ZScale stretch
  (both images share the same `vmin` / `vmax` for a fair visual comparison).
- Difference image: diverging colourmap (`RdBu_r`) centred on zero,
  so that positive residuals appear red and negative residuals appear blue.


- **Author :** Sylvie Dagoret-Campagne
- **Creation date :** 2026-06-12
- **Last update :** 2026-06-13
- **Affiliation :** IJCLab / IN2P3 / CNRS – Université Paris-Saclay
- **Context :** Rubin/LSST sky-alert dipole artifact analysis


In [ ]:
# Standard library
import io
import os

# HTTP client
import requests

# Data wrangling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize, TwoSlopeNorm

# Scipy utilities (kept for potential future use)
from scipy.ndimage import maximum_filter, label, rotate
from scipy.optimize import curve_fit
from scipy.interpolate import griddata

# Astropy – FITS, WCS, coordinates, visualisation
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, AltAz, EarthLocation, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.visualization import ZScaleInterval
from astropy.visualization.mpl_normalize import ImageNormalize

In [ ]:
# ---------------------------------------------------------------------------
# Rubin Observatory site constants (ITRF coordinates)
# ---------------------------------------------------------------------------
RUBIN_LAT_DEG = -30.244728  # geodetic latitude  [deg]
RUBIN_LON_DEG = -70.749417  # east longitude     [deg]
RUBIN_HEIGHT_M = 2647.0  # elevation above sea level [m]

In [ ]:
def parallactic_angle(obs_time, ra, dec, location):
    """Compute the parallactic angle q at (ra, dec) for a given observation.

    Parameters
    ----------
    obs_time : astropy.time.Time
    ra, dec  : astropy.units.Quantity   (angle; passed as Quantity with unit)
    location : astropy.coordinates.EarthLocation

    Returns
    -------
    q : astropy.units.Quantity  (angle, radians)
        Positive when the zenith is to the east of north.
    """
    target = SkyCoord(ra=ra, dec=dec, unit=u.rad)

    # Local sidereal time → hour angle
    lst = obs_time.sidereal_time("apparent", longitude=location.lon)
    ha = (lst - target.ra).to(u.rad)

    lat = location.lat.to(u.rad)
    dec = target.dec.to(u.rad)

    # Standard formula (Meeus, Astronomical Algorithms)
    q = np.arctan2(np.sin(ha), np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(ha))
    return q

## WCS pixel → sky transformation

The FITS WCS linear transform relates pixel offsets to sky offsets via the PC matrix:

$$\begin{pmatrix} \Delta \alpha \\ \Delta \delta \end{pmatrix} = \mathbf{PC} \cdot \begin{pmatrix} x - CRPIX1 \\ y - CRPIX2 \end{pmatrix}$$

Directions (North, East, Zenith, Dipole) are projected into pixel space
using `WCS.world_to_pixel_values` applied to small sky offsets from the image centre.

**Dipole angle convention:** `r:dipoleAngle` is the position angle of the dipole
axis measured from North toward East (standard astronomical PA convention, in degrees).
It is used directly without any offset transformation.


In [ ]:
def plot_cutout_wcs_with_directions(
    fits_file,
    dipole_angle_deg=None,
    dipole_length_pix=None,
    dipole_amplify=6.0,
    cmap="gray",
    diff_image=False,
    zscale=True,
    vmin=None,
    vmax=None,
):
    """Display a FITS cutout with overlaid direction arrows and an optional dipole segment.

    Direction overlays
    ------------------
    - Red    arrow  : celestial North
    - Blue   arrow  : celestial East
    - Orange arrow  : toward Zenith (= DCR displacement direction)
    - Cyan   segment (double-headed) : dipole axis from r:dipoleAngle

    Parameters
    ----------
    fits_file : str
        Path to a FITS file containing WCS, MJD-OBS and observatory keywords.
    dipole_angle_deg : float or None
        ``r:dipoleAngle`` from the Fink alert [degrees, North-toward-East PA].
        The axis is drawn as a symmetric segment through the image centre.
    dipole_length_pix : float or None
        ``r:dipoleLength`` [pixels].  The displayed half-length is
        ``dipole_length_pix / 2 * dipole_amplify``.  When None, a default
        length of 30 % of the image half-size is used.
    dipole_amplify : float
        Amplification factor applied to ``dipole_length_pix`` so that the
        segment is clearly visible even for sub-pixel dipoles (default: 6).
    cmap : str
        Matplotlib colormap (default: 'gray').
    diff_image : bool
        If True, use diverging colourmap RdBu_r centred on zero (overrides cmap).
    zscale : bool
        Apply ZScale stretch (default True).  Ignored when diff_image=True
        or when vmin/vmax are provided.
    vmin, vmax : float or None
        Explicit colour limits.  Override ZScale when provided.
    """
    # ------------------------------------------------------------------
    # Load FITS data and header
    # ------------------------------------------------------------------
    with fits.open(fits_file) as hdul:
        data = hdul[0].data.copy()
        hdr = hdul[0].header.copy()

    wcs = WCS(hdr)

    # ------------------------------------------------------------------
    # Observation time (robust fallback to TAI)
    # ------------------------------------------------------------------
    timesys = str(hdr.get("TIMESYS", "tai")).lower()
    obstime = Time(hdr["MJD-OBS"], format="mjd", scale=timesys)

    # ------------------------------------------------------------------
    # Observatory location (header keywords, fallback to Rubin constants)
    # ------------------------------------------------------------------
    loc = EarthLocation(
        lat=hdr.get("OBS-LAT", RUBIN_LAT_DEG) * u.deg,
        lon=hdr.get("OBS-LONG", RUBIN_LON_DEG) * u.deg,
        height=hdr.get("OBS-ELEV", RUBIN_HEIGHT_M) * u.m,
    )

    # Telescope rotator angle (informational, may be absent)
    rotpa = hdr.get("ROTPA", np.nan) * u.deg

    # ------------------------------------------------------------------
    # Image geometry: centre pixel
    # ------------------------------------------------------------------
    ny, nx = data.shape
    x0, y0 = nx / 2.0, ny / 2.0

    ra0, dec0 = wcs.pixel_to_world_values(x0, y0)
    sky_center = SkyCoord(ra=ra0 * u.deg, dec=dec0 * u.deg, frame="icrs")

    # ------------------------------------------------------------------
    # Local tangent-plane frame at image centre
    # Longitude axis = East, latitude axis = North
    # ------------------------------------------------------------------
    local_frame = SkyOffsetFrame(origin=sky_center)
    north_sky = SkyCoord(0 * u.arcsec, +10 * u.arcsec, frame=local_frame).icrs
    east_sky = SkyCoord(+10 * u.arcsec, 0 * u.arcsec, frame=local_frame).icrs

    # ------------------------------------------------------------------
    # Altitude / azimuth and parallactic angle
    # ------------------------------------------------------------------
    altaz = sky_center.transform_to(AltAz(obstime=obstime, location=loc))
    alt = altaz.alt
    az = altaz.az
    zenith_dist = 90 * u.deg - alt
    q_rad = parallactic_angle(obstime, sky_center.ra, sky_center.dec, loc).to_value(u.rad)
    q_deg = np.degrees(q_rad)

    print(f"Sky center        : RA={ra0:.5f} deg, Dec={dec0:.5f} deg")
    print(f"Altitude          : {alt:.3f}")
    print(f"Zenith distance   : {zenith_dist:.3f}")
    print(f"Azimuth           : {az:.3f}")
    print(f"Parallactic angle : {q_deg:.3f} deg")
    print(f"ROTPA             : {rotpa}")

    # ------------------------------------------------------------------
    # Project sky probes into pixel-offset vectors
    # ------------------------------------------------------------------
    def world_to_vec(coord):
        """Return pixel offset (dx, dy) from image centre for a sky coordinate."""
        x, y = wcs.world_to_pixel_values(coord.ra.deg, coord.dec.deg)
        return x - x0, y - y0

    vx_n, vy_n = world_to_vec(north_sky)
    vx_e, vy_e = world_to_vec(east_sky)

    # Zenith: rotate the North unit vector by the parallactic angle q toward East
    vx_z = np.cos(q_rad) * vx_n + np.sin(q_rad) * vx_e
    vy_z = np.cos(q_rad) * vy_n + np.sin(q_rad) * vy_e

    # ------------------------------------------------------------------
    # Normalise cardinal arrows to arrow_frac * half-image size
    # ------------------------------------------------------------------
    arrow_frac = 0.38
    scale = arrow_frac * min(nx, ny) / 2.0
    mag_n = np.hypot(vx_n, vy_n)
    mag_e = np.hypot(vx_e, vy_e)
    mag_z = np.hypot(vx_z, vy_z)

    dx_n, dy_n = vx_n / mag_n * scale, vy_n / mag_n * scale
    dx_e, dy_e = vx_e / mag_e * scale, vy_e / mag_e * scale
    dx_z, dy_z = vx_z / mag_z * scale, vy_z / mag_z * scale

    aw = max(0.3, scale * 0.04)  # shaft width
    hw = aw * 3  # head width
    hl = aw * 4  # head length

    # ------------------------------------------------------------------
    # Dipole axis segment
    # r:dipoleAngle is the PA (North toward East) in degrees.
    # The dipole is an axis (not a vector), drawn symmetrically.
    # r:dipoleLength is typically only a few pixels, so we amplify it
    # by dipole_amplify to make it visible on the cutout stamp.
    # A minimum of 25 % of the image half-size is guaranteed.
    # ------------------------------------------------------------------
    has_dipole = dipole_angle_deg is not None
    if has_dipole:
        pa_rad = np.radians(dipole_angle_deg % 360.0)
        # Unit vector along the dipole axis in the N/E pixel basis
        # PA = 0 → North, PA = 90 → East
        ux_dip = np.sin(pa_rad) * vx_e / mag_e + np.cos(pa_rad) * vx_n / mag_n
        uy_dip = np.sin(pa_rad) * vy_e / mag_e + np.cos(pa_rad) * vy_n / mag_n

        # Compute the displayed half-length:
        #   use the actual dipole length amplified, but never less than 25% of scale
        if dipole_length_pix is not None:
            half_len = max(dipole_length_pix / 2.0 * dipole_amplify, 0.25 * scale)
        else:
            half_len = 0.30 * scale  # fallback when dipole length is not available

        print(
            f"Dipole half-length displayed : {half_len:.1f} pix  "
            f"(r:dipoleLength={dipole_length_pix}, amplify x{dipole_amplify})"
        )

    # ------------------------------------------------------------------
    # Colour normalisation
    # ------------------------------------------------------------------
    zscale = ZScaleInterval()
    vmin, vmax = zscale.get_limits(data)

    if diff_image:
        # Symmetric diverging scale centred on zero (99th percentile of |data|)
        vlim = np.nanpercentile(np.abs(data), 99)
        norm = TwoSlopeNorm(vmin=-vlim, vcenter=0.0, vmax=vlim)
        # vlim = np.max(np.abs(vmin),np.abs(vmax))
        # norm = Normalize(vmin=-vlim, vmax=vlim)
        used_cmap = "RdBu_r"  # red = positive, blue = negative
    elif vmin is not None and vmax is not None:
        # Caller-supplied limits (e.g. shared range across Science & Template)
        norm = Normalize(vmin=vmin, vmax=vmax)
        used_cmap = cmap
    elif zscale:
        interval = ZScaleInterval()
        vmin_z, vmax_z = interval.get_limits(data)
        norm = Normalize(vmin=vmin_z, vmax=vmax_z)
        used_cmap = cmap
    else:
        norm = None
        used_cmap = cmap

    # ------------------------------------------------------------------
    # Figure
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(projection=wcs)

    im = ax.imshow(data, origin="lower", cmap=used_cmap, norm=norm)
    plt.colorbar(im, ax=ax, label="Pixel value", fraction=0.046, pad=0.04)

    ax.coords.grid(True, color="white", ls="dotted", lw=0.8, alpha=0.6)
    ax.set_xlabel("Right Ascension", fontsize=11)
    ax.set_ylabel("Declination", fontsize=11)

    # Cardinal and zenith arrows in pixel coordinates
    kw_arr = dict(
        width=aw,
        head_width=hw,
        head_length=hl,
        length_includes_head=True,
        transform=ax.get_transform("pixel"),
    )
    # ax.arrow(x0, y0, dx_n, dy_n, color="tomato",     **kw_arr)
    # ax.arrow(x0, y0, dx_e, dy_e, color="dodgerblue", **kw_arr)
    # ax.arrow(x0, y0, dx_z, dy_z, color="darkorange", **kw_arr)
    ax.arrow(x0, y0, dx_n, dy_n, color="red", **kw_arr)
    ax.arrow(x0, y0, dx_e, dy_e, color="blue", **kw_arr)
    ax.arrow(x0, y0, dx_z, dy_z, color="yellow", **kw_arr)

    # Labels just beyond the arrow tips
    loff = 1.18
    kw_txt = dict(
        fontsize=11, fontweight="bold", ha="center", va="center", transform=ax.get_transform("pixel")
    )
    # ax.text(x0 + dx_n * loff, y0 + dy_n * loff, "N", color="tomato",     **kw_txt)
    # ax.text(x0 + dx_e * loff, y0 + dy_e * loff, "E", color="dodgerblue", **kw_txt)
    # ax.text(x0 + dx_z * loff, y0 + dy_z * loff, "Z", color="darkorange", **kw_txt)
    ax.text(x0 + dx_n * loff, y0 + dy_n * loff, "N", color="red", **kw_txt)
    ax.text(x0 + dx_e * loff, y0 + dy_e * loff, "E", color="blue", **kw_txt)
    ax.text(x0 + dx_z * loff, y0 + dy_z * loff, "Z", color="yellow", **kw_txt)

    # Dipole axis: double-headed segment through the centre (ax.annotate <->)
    if has_dipole:
        x_tip1 = x0 + ux_dip * half_len
        y_tip1 = y0 + uy_dip * half_len
        x_tip2 = x0 - ux_dip * half_len
        y_tip2 = y0 - uy_dip * half_len

        pix_tf = ax.get_transform("pixel")
        ax.annotate(
            "",
            xy=(x_tip1, y_tip1),
            xycoords=pix_tf,
            xytext=(x_tip2, y_tip2),
            textcoords=pix_tf,
            arrowprops=dict(
                arrowstyle="<->",
                color="cyan",
                lw=2.5,
                mutation_scale=hw * 4,
            ),
        )
        # Label beyond one tip
        gap = max(3.0, 0.06 * scale)  # small gap between tip and text
        ax.text(
            x_tip1 + ux_dip * gap,
            y_tip1 + uy_dip * gap,
            f"Dipole\nPA={dipole_angle_deg:.1f}°",
            color="cyan",
            fontsize=8,
            fontweight="bold",
            ha="center",
            va="center",
            transform=ax.get_transform("pixel"),
        )

    # Legend
    patches = [
        mpatches.Patch(color="tomato", label="North"),
        mpatches.Patch(color="dodgerblue", label="East"),
        mpatches.Patch(color="darkorange", label=f"Zenith  (q={q_deg:.1f}°)"),
    ]
    if has_dipole:
        patches.append(
            mpatches.Patch(
                color="cyan",
                label=(
                    f"Dipole axis  (PA={dipole_angle_deg:.1f}°, "
                    f"L={dipole_length_pix:.1f} pix × {dipole_amplify:.0f})"
                    if dipole_length_pix is not None
                    else f"Dipole axis  (PA={dipole_angle_deg:.1f}°)"
                ),
            )
        )
    ax.legend(
        handles=patches, loc="lower right", fontsize=8, framealpha=0.65, facecolor="k", labelcolor="white"
    )

    # Informative title
    fname_base = os.path.basename(fits_file)
    ax.set_title(
        f"{fname_base}\n"
        f"MJD={hdr.get('MJD-OBS', float('nan')):.4f}  "
        f"Alt={alt.to_value(u.deg):.1f}°  "
        f"z={zenith_dist.to_value(u.deg):.1f}°  "
        f"q={q_deg:.1f}°",
        fontsize=9,
        pad=8,
    )

    plt.tight_layout()
    plt.show()

## Object selection

Dictionary of COSMOS Deep Drilling Field diaObjects ranked by dipole fraction.
Change `DIAOBJECT_IDX` to switch between objects.


In [ ]:
# Candidate diaObjects in the COSMOS DDF, ranked by dipole fraction
objsid = {
    # 0: 313888627167330394,   # rank 1  – COSMOS, dipole_frac=0.962, n_dipoles=507, mostly before 2024-03
    0: 313985344866353157,  # rank 2  – COSMOS, positive & negative fluxes, small dipoles (many bands)
    1: 313853517840777344,  # rank 3  – COSMOS, positive & negative fluxes, small dipoles
    2: 313972182542712999,  # rank 4  – COSMOS, positive & negative fluxes, range of dipole lengths
    3: 313871013109563545,  # rank 5  – COSMOS, positive & negative fluxes, range of dipole lengths
    4: 313871013420466334,  # rank 6  – COSMOS, positive & negative fluxes, mixed dipole sizes
    5: 313998569477505082,  # rank 7  – COSMOS, positive & negative fluxes
    6: 313994141002367046,  # rank 8  – COSMOS, QSO, two dipole populations in g band
    7: 313888627167330394,  # rank 1  – same as commented rank-1 above
}

In [ ]:
DIAOBJECT_IDX = 0  # ← change this index to select a different object
DIAOBJECT_ID = objsid[DIAOBJECT_IDX]
print(f"Selected diaObjectId: {DIAOBJECT_ID}  (index {DIAOBJECT_IDX})")

## 1 – Fetch the diaSource table from Fink

Retrieve all diaSources associated to the selected diaObject, including
flux columns and dipole characterisation columns (`r:dipoleAngle`, `r:dipoleLength`, `r:isDipole`).


In [ ]:
diaObjectId = str(DIAOBJECT_ID)

# Request all relevant source columns plus dipole characterisation
r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/sources",
    json={
        "diaObjectId": diaObjectId,
        "columns": (
            "r:diaSourceId, r:midpointMjdTai, r:ra, r:raErr, r:dec, r:decErr, "
            "r:apFlux, r:apFluxErr, r:scienceFlux, r:scienceFluxErr, "
            "r:templateFlux, r:templateFluxErr, r:band, "
            "r:dipoleAngle, r:dipoleLength, r:isDipole"
        ),
    },
)

if r.status_code == 200:
    data_sources = pd.read_json(io.BytesIO(r.content))
else:
    raise RuntimeError(f"Fink API error for diaObjectId {diaObjectId}: {r.status_code} – {r.text}")

data_sources.to_csv(f"data_{diaObjectId}.csv", index=False)
display(data_sources.head(10))

In [ ]:
# Quick sanity-check: scienceFlux light curve, one colour per photometric band
fig, ax = plt.subplots(figsize=(11, 4))
for band, grp in data_sources.groupby("r:band"):
    ax.errorbar(
        grp["r:midpointMjdTai"],
        grp["r:scienceFlux"],
        yerr=grp["r:scienceFluxErr"],
        fmt="o",
        ms=4,
        label=band,
        alpha=0.7,
    )
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("MJD (TAI)", fontsize=11)
ax.set_ylabel("scienceFlux [nJy]", fontsize=11)
ax.set_title(f"Light curve – diaObjectId: {diaObjectId}", fontsize=12)
ax.legend(title="band", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 2 – Download FITS cutouts for the first diaSource

For the first entry in the source table, download the Science, Template
and Difference cutouts from Fink, inject the missing time/location
keywords, and save to FITS files on disk.


In [ ]:
for row_idx in range(len(data_sources["r:diaSourceId"])):
    src = str(data_sources["r:diaSourceId"][row_idx])
    mjd_val = data_sources["r:midpointMjdTai"][row_idx]

    obstime_row = Time(mjd_val, format="mjd", scale="tai")
    mjd_str = str(mjd_val).replace(".", "_")

    print(f"Processing diaSourceId: {src}   MJD: {mjd_val:.6f}")

    for kind in ["Science", "Template", "Difference"]:
        r_cut = requests.post(
            "https://api.lsst.fink-portal.org/api/v1/cutouts",
            json={"diaSourceId": src, "kind": kind, "output-format": "FITS"},
        )

        if r_cut.status_code == 200 and len(r_cut.content) > 0:
            try:
                with fits.open(io.BytesIO(r_cut.content), ignore_missing_simple=True) as data_cut:
                    hdr = data_cut[0].header

                    # Inject observation time (needed for parallactic angle)
                    hdr["MJD-OBS"] = (mjd_val, "Observation midpoint [MJD, TAI]")
                    hdr["TIMESYS"] = ("TAI", "Time system")
                    hdr["DATE-OBS"] = (obstime_row.utc.isot, "UTC ISO observation time")
                    hdr["COMMENT"] = "Time keywords injected from Fink midpointMjdTai"

                    # Inject Rubin Observatory site coordinates
                    hdr["OBS-LAT"] = (RUBIN_LAT_DEG, "Rubin latitude  [deg]")
                    hdr["OBS-LONG"] = (RUBIN_LON_DEG, "Rubin longitude [deg]")
                    hdr["OBS-ELEV"] = (RUBIN_HEIGHT_M, "Rubin elevation [m]")

                    filename = f"{mjd_str}_cutout_{kind}.fits"
                    data_cut.writeto(filename, overwrite=True)
                    print(f"  Saved {filename}")

            except Exception as exc:
                print(f"  ERROR diaSourceId={src} kind={kind}: {exc}")
        else:
            print(f"  No content for diaSourceId={src} kind={kind} (HTTP {r_cut.status_code})")

    break  # process only the first diaSource

In [ ]:
# Build FITS file paths for the three cutout types
selected_prefix = f"{mjd_str}_cutout"
file_sci = f"{selected_prefix}_Science.fits"
file_temp = f"{selected_prefix}_Template.fits"
file_diff = f"{selected_prefix}_Difference.fits"
print("Science   :", file_sci)
print("Template  :", file_temp)
print("Difference:", file_diff)

## 3 – Inspect the Difference FITS header

Verify that WCS and time keywords have been correctly injected.


In [ ]:
with fits.open(file_diff) as hdu:
    header_diff = hdu[0].header
    img_diff = hdu[0].data

display(header_diff)

In [ ]:
# Print the WCS object to confirm pixel scale and orientation
print(WCS(header_diff))

In [ ]:
# Raw preview of the Difference image in pixel coordinates (no WCS projection)
vlim = np.nanpercentile(np.abs(img_diff), 99)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(img_diff, origin="lower", cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-vlim, vcenter=0.0, vmax=vlim))
plt.colorbar(im, ax=ax, label="Pixel value")
ax.set_title("Difference cutout – raw pixel values (red>0, blue<0)")
plt.tight_layout()
plt.show()

## 4 – Extract dipole parameters for the first diaSource

`r:dipoleAngle` is the position angle of the dipole axis measured from North toward
East [degrees].  It is used directly (no offset).

`r:dipoleLength` is the separation between the two dipole lobes in pixels.  Because
this value is often only a few pixels (much smaller than the stamp size), the drawn
segment is amplified by `dipole_amplify` (default ×6) so it remains clearly visible.


In [ ]:
# Use the first row (same source for which cutouts were downloaded)
first_row = data_sources.iloc[0]

dipole_angle_deg = None
dipole_length_pix = None

if "r:dipoleAngle" in first_row and pd.notna(first_row["r:dipoleAngle"]):
    dipole_angle_deg = float(first_row["r:dipoleAngle"])
    print(f"r:dipoleAngle  = {dipole_angle_deg:.2f} deg  (PA: North toward East)")
else:
    print("r:dipoleAngle not available for this source – dipole axis will not be drawn.")

if "r:dipoleLength" in first_row and pd.notna(first_row["r:dipoleLength"]):
    dipole_length_pix = float(first_row["r:dipoleLength"])
    print(f"r:dipoleLength = {dipole_length_pix:.2f} pixels")

## 5 – Display Difference cutout with WCS directions and dipole axis

Diverging colourmap (`RdBu_r`): **red = positive flux**, **blue = negative flux**,
centred on zero using `TwoSlopeNorm`.

The dipole segment is amplified (`dipole_amplify`) to be clearly visible.
Adjust the factor in the call below if needed.


In [ ]:
DIPOLE_AMPLIFY = 6  # ← increase to make the dipole segment longer

plot_cutout_wcs_with_directions(
    file_diff,
    dipole_angle_deg=dipole_angle_deg,
    dipole_length_pix=dipole_length_pix,
    dipole_amplify=DIPOLE_AMPLIFY,
    diff_image=True,  # → RdBu_r diverging cmap centred on zero
)

## 6 – Science and Template cutouts (for reference)

Grey scale with a **shared ZScale colour range** computed on the concatenation of
both images, so that pixel values are directly comparable between the two panels.


In [ ]:
# Load both images
with fits.open(file_sci) as hdu:
    img_sci = hdu[0].data
with fits.open(file_temp) as hdu:
    img_temp = hdu[0].data

# Compute a shared ZScale stretch on the concatenated pixel stack
combined = np.concatenate([img_sci.ravel(), img_temp.ravel()])
combined_clean = combined[np.isfinite(combined)]
vmin_shared, vmax_shared = ZScaleInterval().get_limits(combined_clean)
print(f"Shared colour range: vmin={vmin_shared:.3f}, vmax={vmax_shared:.3f}")

# Display side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, img, label_str in zip(
    axes,
    [img_sci, img_temp],
    ["Science", "Template"],
):
    im = ax.imshow(img, origin="lower", cmap="gray", vmin=vmin_shared, vmax=vmax_shared)
    plt.colorbar(im, ax=ax, label="Pixel value")
    ax.set_title(f"{label_str} – MJD {mjd_val:.4f}", fontsize=10)
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")

plt.suptitle(
    f"diaObjectId: {diaObjectId}  –  shared ZScale range [{vmin_shared:.1f}, {vmax_shared:.1f}]",
    fontsize=11,
    y=1.01,
)
plt.tight_layout()
plt.show()